# Small Kiva graph example

This notebook draws a tiny sample of the graph so we can see the structure before training the full GNN.

- Blue nodes: loans
- Orange nodes: field partners
- Green nodes: regions
- Purple nodes: borrower-gender groups
- Solid lines: loan-partner target links
- Dashed lines: loan context links

In [ ]:
import csv
from pathlib import Path
import matplotlib.pyplot as plt

# Read only a small sample; the complete graph is too large to draw.
data_file = Path('../data/processed/loans_joined.csv')
sample_size = 12
rows = []
with data_file.open(encoding='utf-8', errors='replace', newline='') as f:
    for row in csv.DictReader(f):
        if row.get('loan_id') and row.get('partner_id'):
            rows.append(row)
        if len(rows) == sample_size:
            break

print(f'Loans shown: {len(rows)}')
print('Partners:', sorted({r['partner_id'] for r in rows}))
print('Regions:', sorted({r['loan_region'] or 'unknown' for r in rows}))
print('Genders:', sorted({r['borrower_gender'] or 'unknown' for r in rows}))

In [ ]:
# Create a small bipartite/context graph by hand so every edge is visible.
loan_nodes = [f"loan:{r['loan_id']}" for r in rows]
partner_nodes = [f"partner:{r['partner_id']}" for r in rows]
region_nodes = [f"region:{r['loan_region'] or 'unknown'}" for r in rows]
gender_nodes = [f"gender:{r['borrower_gender'] or 'unknown'}" for r in rows]
partners = list(dict.fromkeys(partner_nodes))
regions = list(dict.fromkeys(region_nodes))
genders = list(dict.fromkeys(gender_nodes))

# Place node groups in four vertical columns.
positions = {}
for i, node in enumerate(loan_nodes): positions[node] = (0, -i)
for i, node in enumerate(partners): positions[node] = (2, -i * 2)
for i, node in enumerate(regions): positions[node] = (4, -i * 1.5)
for i, node in enumerate(genders): positions[node] = (6, -i * 3)

fig, ax = plt.subplots(figsize=(14, 8))
colors = {'loan': '#2563eb', 'partner': '#f97316', 'region': '#16a34a', 'gender': '#9333ea'}

for row in rows:
    loan = f"loan:{row['loan_id']}"
    partner = f"partner:{row['partner_id']}"
    region = f"region:{row['loan_region'] or 'unknown'}"
    gender = f"gender:{row['borrower_gender'] or 'unknown'}"
    x1, y1 = positions[loan]
    # Solid edge is the target relationship the GNN predicts.
    x2, y2 = positions[partner]
    ax.plot([x1, x2], [y1, y2], color='#64748b', linewidth=1.8)
    # Dashed edges provide known context for the loan representation.
    for context in (region, gender):
        x2, y2 = positions[context]
        ax.plot([x1, x2], [y1, y2], color='#cbd5e1', linestyle='--', linewidth=1)

for node, (x, y) in positions.items():
    kind = node.split(':', 1)[0]
    ax.scatter(x, y, s=520, color=colors[kind], edgecolor='white', linewidth=1.5, zorder=3)
    label = node if len(node) < 25 else node[:22] + '...'
    ax.text(x + .08, y, label, va='center', fontsize=8)

ax.set_title('Small Kiva heterogeneous graph sample')
ax.set_xlim(-1, 8); ax.axis('off'); plt.show()

## What this shows

The GNN receives the loan's own features and information from its neighbors. The solid loan-partner edge is the link-prediction target. Region and gender edges are context: they help create a useful loan embedding but are not the target being predicted. In the full graph there are hundreds of thousands of these relationships, so we inspect a small sample here.

## Objective function and model output

For a loan node `u` and partner node `v`, the GNN creates embeddings `z_u` and `z_v`. The link score is their dot product, `s(u,v) = z_u · z_v`. Applying a sigmoid turns that score into a link probability.

The training label is **1** when the loan-partner edge was observed and **0** for a sampled loan-partner pair that was not observed. The objective is binary cross-entropy:

`L = -mean[y log(sigmoid(s)) + (1-y) log(1-sigmoid(s))]`

Therefore, the current model output means: **how likely is this partner to be connected/recommended for this loan?** It does not mean whether the loan will succeed, be fully funded, or be repaid. Predicting loan success would be a different supervised-learning task with a different label, such as a carefully defined funded/repayment outcome.

## Link prediction walkthrough

The next cell is a small visual demonstration of the prediction step. We pretend that one loan's known partner edge is hidden, calculate a score for several candidate partners, and rank them. The numbers are illustrative embeddings, not a training result.

In [ ]:
import numpy as np

# Fixed example embeddings make this demonstration repeatable.
loan_embedding = np.array([0.8, 0.1, 0.6, 0.2])
partner_embeddings = {
    'partner A': np.array([0.7, 0.2, 0.5, 0.1]),
    'partner B': np.array([0.1, 0.9, 0.1, 0.4]),
    'partner C': np.array([0.6, 0.0, 0.7, 0.2]),
    'partner D': np.array([0.2, 0.3, 0.2, 0.9]),
}

# The decoder compares the loan embedding with each partner embedding.
names = list(partner_embeddings)
scores = [float(loan_embedding @ partner_embeddings[name]) for name in names]
probabilities = [1 / (1 + np.exp(-score)) for score in scores]
ranking = sorted(zip(names, probabilities), key=lambda item: item[1], reverse=True)

for rank, (name, probability) in enumerate(ranking, start=1):
    print(f'{rank}. {name}: predicted link probability = {probability:.3f}')

plt.figure(figsize=(8, 4))
plt.bar([item[0] for item in ranking], [item[1] for item in ranking], color='#f97316')
plt.ylim(0, 1)
plt.ylabel('sigmoid(dot-product score)')
plt.title('Illustrative partner ranking for one loan')
plt.show()